# RNA → Protein Multimodal Autoencoder
### Multi-omics integration notebook for factorial time-course data
**Design:** 2 varieties × 2 treatments × 4 timepoints × 3 replicates = 48 samples  
**Data:** ~60,000 genes · ~6,000 proteins  

---
Work through each section in order. Every section shows:
- the **parameters** you can tweak in the CONFIG dict  
- the **code** that runs  
- **inline figures** and results  

Start with `--demo` (synthetic data, known ground truth) before loading your real files.


## 0 · Setup

In [ ]:
# ── install once (comment out after first run) ───────────────────────────────
# !pip install torch scikit-learn pandas numpy scipy matplotlib seaborn -q

import sys, os, warnings
warnings.filterwarnings("ignore")

# make the rnaprot package importable from this notebook
PROJ = os.path.dirname(os.path.abspath("__file__"))
if PROJ not in sys.path:
    sys.path.insert(0, PROJ)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from IPython.display import display
import torch

sns.set_theme(style="ticks", font_scale=1.05)
SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"Python {sys.version.split()[0]}  |  torch {torch.__version__}  |  pandas {pd.__version__}")


## 1 · Global configuration

Adjust these before running anything else. Each parameter is explained inline.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#                        ← EDIT THESE ←
# ═══════════════════════════════════════════════════════════════════════════

# ── Data paths (leave as None to run on synthetic demo data) ───────────────
RNA_CSV     = None   # e.g. "data/rna_counts.csv"    rows=genes,  cols=samples
PROT_CSV    = None   # e.g. "data/protein_lfq.csv"   rows=prots,  cols=samples
META_CSV    = None   # e.g. "data/metadata.csv"       cols: variety,treatment,timepoint,replicate
MAPPING_CSV = None   # e.g. "data/protein_to_gene.csv"  cols: protein_id, gene_id

# ── Preprocessing ──────────────────────────────────────────────────────────
N_RNA       = 3000   # genes kept after variance filtering (of your ~60k)
N_PROT      = 1500   # proteins kept (of your ~6k); raise once you're happy
RNA_MODE    = "counts"      # "counts" → CPM+log2  |  "logged" → already normalised
PROT_MODE   = "intensity"   # "intensity" → log2   |  "logged" → already logged

# ── Autoencoder architecture ───────────────────────────────────────────────
LATENT      = 12     # latent dimensions — do NOT exceed ~16 at n=48
HIDDEN      = (128, 32)   # encoder layer sizes (decoder is the mirror)
DROPOUT     = 0.15
EPOCHS      = 400
PATIENCE    = 60     # early-stopping patience (epochs without improvement)
LR          = 1e-3
WEIGHT_DECAY = 1e-3

# ── Loss weights ───────────────────────────────────────────────────────────
# cross_weight is the RNA→protein term — the one that matters most
W_RECON     = 0.5    # weight on RNA→RNA and protein→protein reconstruction
W_CROSS     = 3.0    # weight on RNA→protein (cross-modal prediction)
W_ALIGN     = 0.5    # weight on ||z_RNA − z_protein||
VARIATIONAL = False  # True → VAE (adds a KL smoothness prior)

# ── Cross-validation ───────────────────────────────────────────────────────
SPLIT_SCHEME = "leave_condition_out"  # honest default
# options: "leave_condition_out" | "leave_variety_out" | "leave_timepoint_out" | "naive_random"

N_PERM      = 2      # label-permutation nulls (≥5 for publication, 2 is fast)

# ═══════════════════════════════════════════════════════════════════════════
OUT_DIR = Path("results_notebook")
OUT_DIR.mkdir(exist_ok=True)
(OUT_DIR / "figures").mkdir(exist_ok=True)

print("Config loaded ✓")
print(f"  latent={LATENT}, hidden={HIDDEN}, cross_weight={W_CROSS}")
print(f"  split={SPLIT_SCHEME}, n_genes={N_RNA}, n_prot={N_PROT}")


## 2 · Load data

If `RNA_CSV` is `None` the notebook uses **synthetic data** with seven planted
protein classes and known ground truth — a built-in positive control.  
Set the path variables in section 1 to switch to your real files.


In [ ]:
from rnaprot.simulate import simulate
from rnaprot.data import load_from_csv

USE_DEMO = (RNA_CSV is None)

if USE_DEMO:
    data, truth = simulate(seed=SEED)
    print("Running on SYNTHETIC demo data — ground truth classes are known.")
    print("Set RNA_CSV etc. in section 1 to use your real data.")
else:
    data = load_from_csv(RNA_CSV, PROT_CSV, META_CSV, MAPPING_CSV)
    truth = None
    print("Running on REAL data.")

print()
print(data.describe())


In [ ]:
# ── Visualise the experimental design ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))

meta = data.meta

for ax, col, title in zip(axes,
                           ["variety", "treatment", "timepoint"],
                           ["Varieties", "Treatments", "Timepoints"]):
    counts = meta[col].value_counts().sort_index()
    bars = ax.bar(counts.index, counts.values, color=sns.color_palette("Set2", len(counts)))
    ax.set_title(title, fontweight="bold")
    ax.set_ylabel("Samples")
    ax.set_xlabel(col.capitalize())
    for b in bars:
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.2,
                str(int(b.get_height())), ha="center", fontsize=9)
    ax.spines[["top","right"]].set_visible(False)

fig.suptitle(f"Experimental design  ({data.n_samples} samples)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR / "figures" / "design.png", dpi=150, bbox_inches="tight")
plt.show()


## 3 · Preprocessing & feature selection

**Critical rule:** every filtering and scaling decision is fit on training
samples only. Selecting the top-2000 most variable genes on the full matrix
before cross-validation leaks test information and can manufacture double-digit
R² from pure noise.

The `OmicsPreprocessor` below enforces this — it is always called inside the
CV loop.  This cell shows you the filtering result on the full matrix so you can
sanity-check the numbers before running CV.


In [ ]:
from rnaprot.data import OmicsPreprocessor, covariate_matrix, design_matrix

train_all = np.arange(data.n_samples)
pre_full  = OmicsPreprocessor(n_rna=N_RNA, n_prot=N_PROT,
                               rna_mode=RNA_MODE, prot_mode=PROT_MODE).fit(data, train_all)
R_full, P_full = pre_full.transform(data)
cov_full = covariate_matrix(data.meta)

print(f"RNA  : {data.rna.shape[1]:,} genes \u2192 {R_full.shape[1]:,} after filtering")
print(f"Prot : {data.prot.shape[1]:,} proteins \u2192 {P_full.shape[1]:,} after filtering")
print(f"Covariates: {cov_full.shape[1]} (variety/treatment/timepoint dummies)")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))

for ax, mat, label in zip(axes,
                           [R_full, P_full],
                           [f"RNA (n={R_full.shape[1]:,})", f"Protein (n={P_full.shape[1]:,})"]):
    var = mat.var(axis=0)
    # use Freedman-Diaconis bins, fall back to sqrt rule if degenerate
    try:
        counts, edges = np.histogram(var, bins="fd")
    except Exception:
        n_b = max(2, int(np.sqrt(len(var))) // 2)
        lo, hi = float(var.min()), float(var.max()) + 1e-9
        counts, edges = np.histogram(var, bins=n_b, range=(lo, hi))
    ax.bar(edges[:-1], counts, width=np.diff(edges),
           color=sns.color_palette("Set2")[0], alpha=0.85, align="edge")
    ax.axvline(float(np.median(var)), color="crimson", lw=1.5, ls="--",
               label=f"median = {np.median(var):.2f}")
    ax.set_xlabel("Variance (standardised units)")
    ax.set_ylabel("Features")
    ax.set_title(label, fontweight="bold")
    ax.legend(fontsize=9)
    ax.spines[["top","right"]].set_visible(False)

fig.suptitle("Feature variance distributions after preprocessing", fontsize=11)
plt.tight_layout()
plt.savefig(OUT_DIR / "figures" / "variance_distributions.png", dpi=150, bbox_inches="tight")
plt.show()


## 4 · Cross-validation — all models

Six models run on the same folds:

| model | what it is | why it's here |
|---|---|---|
| `mean` | training mean | defines R² = 0 |
| `design_only` | ridge on variety/treatment/time, **no RNA** | if AE ≈ this, the model learned the design not RNA biology |
| `cognate_ridge` | ridge on the protein's own transcript + design | classical biology baseline |
| `pca_ridge` | PCA on RNA → ridge → all proteins | best linear latent space model |
| `pls` | partial least squares | supervised analogue of DIABLO |
| `multimodal_ae` | cross-modal autoencoder | nonlinear; wins if biology is nonlinear |

The honest split holds out **all 3 replicates of one condition** at once.
`naive_random` is a diagnostic to show how much replicate leakage inflates results.


In [ ]:
from rnaprot.data import SPLIT_SCHEMES, _groups
from rnaprot.models import AERegressor
from rnaprot import baselines as B
from rnaprot.evaluate import run_cv, oof_r2_table, compare_models

# ── cognate index function (maps each retained protein to its transcript col) ─
def make_cognate_index_fn(cognate):
    def fn(pre):
        gpos = {g: i for i, g in enumerate(pre.rna_features_)}
        return np.array([gpos.get(cognate.get(p), -1)
                         for p in pre.prot_features_])
    return fn

cog_fn = make_cognate_index_fn(data.cognate)
grp_all = _groups(data.meta, ("variety", "treatment", "timepoint"))

# ── model factory (rebuilt per fold so features can differ) ──────────────────
def build_models():
    def ae_factory(ctx):
        return AERegressor(
            latent=LATENT, hidden=HIDDEN, dropout=DROPOUT,
            epochs=EPOCHS, patience=PATIENCE, lr=LR,
            weight_decay=WEIGHT_DECAY,
            weights=(W_RECON, W_RECON, W_CROSS, W_ALIGN),
            variational=VARIATIONAL,
            groups=ctx.get("train_groups"),
        )
    return {
        "mean":          lambda ctx: B.MeanBaseline(),
        "design_only":   lambda ctx: B.DesignBaseline(),
        "cognate_ridge": lambda ctx: B.CognateBaseline(ctx["pairs"]),
        "pca_ridge":     lambda ctx: B.PCARidge(n_components=10),
        "pls":           lambda ctx: B.PLSBaseline(n_components=5),
        "multimodal_ae": ae_factory,
    }

# ── run honest CV ──────────────────────────────────────────────────────────
print(f"Running CV: {SPLIT_SCHEME}")
splits = SPLIT_SCHEMES[SPLIT_SCHEME](data.meta)
pre_kw = dict(n_rna=N_RNA, n_prot=N_PROT, rna_mode=RNA_MODE, prot_mode=PROT_MODE)
folds_df, oof = run_cv(data, splits, build_models(), pre_kw, cog_fn, verbose=True)

r2_table = oof_r2_table(oof, data.meta)
folds_df.to_csv(OUT_DIR / "cv_folds.csv", index=False)
r2_table.to_csv(OUT_DIR / f"per_protein_r2_{SPLIT_SCHEME}.csv")

print()
print(folds_df.groupby("model")[["median_r2","mean_r2","frac_r2_pos"]].mean().round(3).to_string())


In [ ]:
# ── Figure: model comparison boxplots ────────────────────────────────────
MODEL_ORDER  = ["mean", "design_only", "cognate_ridge", "pca_ridge", "pls", "multimodal_ae"]
PALETTE      = dict(zip(MODEL_ORDER, sns.color_palette("Set2", 6)))

fig, ax = plt.subplots(figsize=(10, 4.5))
data_box = [r2_table[m].dropna().clip(-1, 1).values for m in MODEL_ORDER if m in r2_table]
labels    = [m for m in MODEL_ORDER if m in r2_table]
bp = ax.boxplot(data_box, tick_labels=labels, patch_artist=True, showfliers=False,
                medianprops=dict(color="black", lw=2))

for patch, lbl in zip(bp["boxes"], labels):
    patch.set_facecolor(PALETTE.get(lbl, "steelblue"))
    patch.set_alpha(0.80)

ax.axhline(0, color="crimson", lw=1.5, ls="--", label="R² = 0 (training mean)")
ax.set_ylabel("Per-protein R² (out-of-fold)", fontsize=11)
ax.set_title(f"Model comparison — {SPLIT_SCHEME}", fontweight="bold", fontsize=12)
ax.tick_params(axis="x", rotation=20)
ax.legend(fontsize=9)
ax.spines[["top","right"]].set_visible(False)

# annotate medians
for i, lbl in enumerate(labels):
    med = r2_table[lbl].dropna().clip(-1, 1).median()
    ax.text(i + 1, med + 0.02, f"{med:.2f}", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.savefig(OUT_DIR / "figures" / "model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Red dashed line = predicting the training mean (R² = 0). Any model below this is worse than useless.")


In [ ]:
# ── Paired comparison vs pca_ridge ───────────────────────────────────────
comp = compare_models(r2_table, reference="pca_ridge")
comp.to_csv(OUT_DIR / "model_comparison.csv", index=False)

display(comp[["model","median_r2","median_ref","median_delta",
              "win_frac","wilcoxon_p","n_proteins"]].round(4))

print()
print("median_delta > 0  → model beats pca_ridge on more than half the proteins")
print("win_frac          → fraction of proteins where model wins")
print("wilcoxon_p        → paired test p-value (many proteins, so treat with care)")


In [ ]:
# ── Leakage demonstration (naive_random vs honest split) ─────────────────
print("Running naive_random CV (diagnostic only — do NOT report this as your result) ...")
splits_naive = SPLIT_SCHEMES["naive_random"](data.meta)
folds_naive, oof_naive = run_cv(data, splits_naive, build_models(), pre_kw, cog_fn, verbose=False)
r2_naive = oof_r2_table(oof_naive, data.meta)
folds_naive.to_csv(OUT_DIR / "cv_folds_naive.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, (r2t, scheme) in zip(axes, [(r2_table, SPLIT_SCHEME), (r2_naive, "naive_random")]):
    data_box = [r2t[m].dropna().clip(-1, 1).values for m in MODEL_ORDER if m in r2t]
    labels   = [m for m in MODEL_ORDER if m in r2t]
    bp = ax.boxplot(data_box, tick_labels=labels, patch_artist=True, showfliers=False,
                    medianprops=dict(color="black", lw=1.5))
    for patch, lbl in zip(bp["boxes"], labels):
        patch.set_facecolor(PALETTE.get(lbl, "steelblue")); patch.set_alpha(0.80)
    ax.axhline(0, color="crimson", lw=1.5, ls="--")
    ax.set_title(scheme, fontweight="bold")
    ax.tick_params(axis="x", rotation=20)
    ax.spines[["top","right"]].set_visible(False)
axes[0].set_ylabel("Per-protein R² (out-of-fold)")
fig.suptitle("Replicate leakage inflates performance (right panel is an artefact)", fontsize=11)
plt.tight_layout()
plt.savefig(OUT_DIR / "figures" / "leakage.png", dpi=150, bbox_inches="tight")
plt.show()
naive_med  = r2_naive["pca_ridge"].dropna().clip(-1,1).median()
honest_med = r2_table["pca_ridge"].dropna().clip(-1,1).median()
print(f"pca_ridge median R²:  honest={honest_med:.3f}  naive={naive_med:.3f}  "
      f"inflation={naive_med - honest_med:.3f}")


## 5 · Permutation null

Shuffles the protein matrix and reruns the entire CV. Any R² surviving this is
a fitting artefact. With n = 48 this is the most convincing control you can put
in a paper.


In [ ]:
from rnaprot.evaluate import permutation_null

print(f"Running {N_PERM} label permutations ...")
null_df = permutation_null(data, SPLIT_SCHEMES[SPLIT_SCHEME](data.meta),
                           build_models(), n_perm=N_PERM,
                           pre_kwargs=pre_kw, cognate_index_fn=cog_fn)
null_df.to_csv(OUT_DIR / "permutation_null.csv")
print(null_df.round(3).to_string())

fig, ax = plt.subplots(figsize=(8, 3.8))
real_meds = {m: r2_table[m].dropna().median() for m in MODEL_ORDER if m in r2_table}
null_meds = null_df.mean(axis=1)   # mean across permutations

x = range(len(real_meds))
ax.bar([i - 0.2 for i in x], real_meds.values(), 0.38,
       label="Real", color=[PALETTE.get(m) for m in real_meds], alpha=0.85)
ax.bar([i + 0.2 for i in x], [null_meds.get(m, 0) for m in real_meds], 0.38,
       label="Permuted", color="lightgray", alpha=0.9)
ax.set_xticks(list(x)); ax.set_xticklabels(list(real_meds), rotation=20, ha="right")
ax.axhline(0, color="crimson", lw=1, ls="--")
ax.set_ylabel("Median R² across proteins")
ax.set_title("Real vs permuted-label performance", fontweight="bold")
ax.legend(); ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.savefig(OUT_DIR / "figures" / "permutation_null.png", dpi=150, bbox_inches="tight")
plt.show()


## 6 · RNA–protein discordance (Question 2)

For every protein: `D = observed − predicted from RNA (out-of-fold)`

Then an F-test for whether D depends systematically on treatment.

- **Large random D** = measurement noise  
- **Structured D that tracks treatment** = post-transcriptional regulation

This is the strongest biological result the pipeline can produce at n = 48.


In [ ]:
from rnaprot.evaluate import discordance_table, classify_kinetics

disc, D = discordance_table(oof, data.meta, model="multimodal_ae")
if USE_DEMO and truth is not None:
    disc = disc.join(truth["truth_class"])
disc.sort_values("q_value").to_csv(OUT_DIR / "discordance.csv")
D.to_csv(OUT_DIR / "discordance_matrix.csv")

n_sig = int((disc["q_value"] < 0.05).sum())
print(f"{n_sig}/{len(disc)} proteins with treatment-dependent residuals at FDR 5%")
print()
print(disc.sort_values("q_value").head(10)[
    ["mean_abs_discordance","F_stat","q_value","explained_var_by_rna"]
    + (["truth_class"] if "truth_class" in disc else [])
].round(3).to_string())


In [ ]:
# ── Volcano plot ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5.5))

x = disc["mean_abs_discordance"]
y = -np.log10(disc["p_value"].clip(1e-300))

if "truth_class" in disc:
    palette = dict(zip(disc["truth_class"].unique(),
                       sns.color_palette("tab10", disc["truth_class"].nunique())))
    for cls, g in disc.groupby("truth_class"):
        ax.scatter(g["mean_abs_discordance"],
                   -np.log10(g["p_value"].clip(1e-300)),
                   s=10, alpha=0.55, label=cls, color=palette[cls])
    ax.legend(title="Planted class", fontsize=8, markerscale=2)
else:
    sig = disc["q_value"] < 0.05
    ax.scatter(x[~sig], y[~sig], s=8, alpha=0.4, color="lightsteelblue", label="not sig")
    ax.scatter(x[sig],  y[sig],  s=10, alpha=0.7, color="crimson", label="FDR < 5%")
    ax.legend(fontsize=9)

ax.axhline(-np.log10(0.05), ls="--", color="grey", lw=1.2, label="p = 0.05")
ax.set_xlabel("|RNA → protein residual| (mean)", fontsize=11)
ax.set_ylabel(r"$-\log_{10}$ p-value", fontsize=11)
ax.set_title("Where RNA stops explaining protein
(treatment-dependent discordance)", fontweight="bold")
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.savefig(OUT_DIR / "figures" / "discordance_volcano.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── Kinetic clusters of residual trajectories ────────────────────────────
N_CLUSTERS = 6   # ← change this

clusters, prof = classify_kinetics(D, data.meta, n_clusters=N_CLUSTERS)
clusters.to_frame().join(disc).to_csv(OUT_DIR / "kinetic_clusters.csv")

fig, axes = plt.subplots(1, N_CLUSTERS, figsize=(2.8 * N_CLUSTERS, 3.2), sharey=True)
pal = sns.color_palette("Set2", N_CLUSTERS)
for i, ax in enumerate(axes):
    sub = prof.loc[clusters[clusters == i].index]
    ax.plot(sub.T.to_numpy(), color="grey", alpha=0.07, lw=0.5)
    ax.plot(sub.mean(axis=0).to_numpy(), color=pal[i], lw=2.5)
    ax.set_title(f"Cluster {i}\n(n={len(sub)})", fontsize=9, fontweight="bold")
    ax.set_xticks(range(prof.shape[1]))
    ax.set_xticklabels(prof.columns, rotation=75, fontsize=7)
    ax.spines[["top","right"]].set_visible(False)
axes[0].set_ylabel("Standardised discordance")
fig.suptitle("Regulatory kinetic classes (residual trajectories)", fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR / "figures" / "kinetic_clusters.png", dpi=150, bbox_inches="tight")
plt.show()

if USE_DEMO and truth is not None:
    print(pd.crosstab(clusters, disc["truth_class"]).to_string())


## 7 · Latent factors — shared axes of variation (Question 1)

Refit the autoencoder on **all 48 samples** and inspect the shared latent space.
Each dimension is annotated with the fraction of its variance explained by each
design term — the neural analogue of MOFA factor annotation.


In [ ]:
from rnaprot.evaluate import latent_design_anova

# ── refit on full data ────────────────────────────────────────────────────
print("Fitting autoencoder on all samples (for interpretation) ...")
ae_full = AERegressor(
    latent=LATENT, hidden=HIDDEN, dropout=DROPOUT, epochs=EPOCHS,
    patience=PATIENCE, lr=LR, weight_decay=WEIGHT_DECAY,
    weights=(W_RECON, W_RECON, W_CROSS, W_ALIGN),
    variational=VARIATIONAL,
    groups=grp_all, verbose=True,
).fit(R_full, P_full, cov_full)

Z = ae_full.encode(R_full, cov_full, "rna")
lat_df = pd.DataFrame(Z, index=data.meta.index,
                      columns=[f"z{i+1}" for i in range(Z.shape[1])])
lat_df.join(data.meta).to_csv(OUT_DIR / "latent_factors.csv")

anova = latent_design_anova(Z, data.meta)
anova.to_csv(OUT_DIR / "latent_design_anova.csv")

print()
print("Variance of each latent factor explained by design terms:")
display(anova.round(3))


In [ ]:
# ── Heatmap: what each latent dim encodes ────────────────────────────────
ann_cols = [c for c in anova.columns if c != "total_R2"]
fig, ax = plt.subplots(figsize=(7, max(4, len(anova) * 0.55)))
im = ax.imshow(anova[ann_cols].to_numpy(), aspect="auto", cmap="YlOrRd",
               vmin=0, vmax=max(0.05, float(anova[ann_cols].max().max())))
ax.set_xticks(range(len(ann_cols))); ax.set_xticklabels(ann_cols, rotation=30, ha="right")
ax.set_yticks(range(len(anova))); ax.set_yticklabels(anova.index)
plt.colorbar(im, ax=ax, label="Variance explained by design term")
ax.set_title("Latent factor × design annotation", fontweight="bold")
for i in range(len(anova)):
    for j, c in enumerate(ann_cols):
        val = anova[ann_cols].iloc[i, j]
        if val > 0.05:
            ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=7.5,
                    color="white" if val > 0.3 else "black")
plt.tight_layout()
plt.savefig(OUT_DIR / "figures" / "latent_design.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── Scatter: top two latent dimensions, coloured by design ───────────────
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, col, title in zip(axes,
                           ["variety", "treatment", "timepoint"],
                           ["Variety", "Treatment", "Timepoint"]):
    groups  = data.meta[col].astype(str).to_numpy()
    palette = dict(zip(sorted(set(groups)), sns.color_palette("Set2", len(set(groups)))))
    for g in sorted(set(groups)):
        idx = groups == g
        ax.scatter(Z[idx, 0], Z[idx, 1], label=g,
                   color=palette[g], s=60, alpha=0.85, edgecolors="white", lw=0.5)
    ax.set_xlabel("z1"); ax.set_ylabel("z2")
    ax.set_title(f"Coloured by {title}", fontweight="bold")
    ax.legend(fontsize=8)
    ax.spines[["top","right"]].set_visible(False)

fig.suptitle("Shared latent space (z1 vs z2)", fontsize=11)
plt.tight_layout()
plt.savefig(OUT_DIR / "figures" / "latent_scatter.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── Training curve ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(ae_full.history_, color="steelblue", lw=2)
ax.axvline(ae_full.epochs_run_ - 1, color="crimson", ls="--", lw=1.5,
           label=f"Early stop (epoch {ae_full.epochs_run_})")
ax.set_xlabel("Epoch"); ax.set_ylabel("Validation cross-modal MSE")
ax.set_title("Autoencoder training (early stopped on RNA→protein path)", fontweight="bold")
ax.legend(fontsize=9); ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.savefig(OUT_DIR / "figures" / "training_curve.png", dpi=150, bbox_inches="tight")
plt.show()


## 8 · Regulatory lag atlas (temporal analysis)

Per protein: compare cognate-ridge performance when RNA_t predicts protein_t
versus when RNA_t predicts protein_{t+1}.

- **RNA-first / protein-later** (`lag_preference > 0.05`): transcript rises first, protein follows
- **synchronous** (`lag_preference < −0.05`): both respond together
- **ambiguous**: noisy or unclear

⚠️ Filter lag hits against the discordance table: `protein_only` proteins also
prefer lag-1 because a monotone temporal trend is trivially easier to predict one step ahead.


In [ ]:
from rnaprot.data import lag_pairs
from rnaprot.evaluate import per_protein_r2

# ── per-protein lag comparison (4-fold CV) ───────────────────────────────
lag_r2 = {}
for lag in (0, 1):
    pairs = (np.stack([np.arange(data.n_samples)] * 2, axis=1) if lag == 0
             else lag_pairs(data.meta, lag))
    pairs_idx = np.array([
        {g: i for i, g in enumerate(pre_full.rna_features_)}.get(
            data.cognate.get(p), -1)
        for p in pre_full.prot_features_
    ])
    Rl, Pl, cl = R_full[pairs[:,0]], P_full[pairs[:,1]], cov_full[pairs[:,0]]
    rng = np.random.default_rng(SEED)
    folds = np.array_split(rng.permutation(len(pairs)), 4)
    acc   = []
    for f in folds:
        tr = np.setdiff1d(np.arange(len(pairs)), f)
        m  = B.CognateBaseline(pairs_idx).fit(Rl[tr], Pl[tr], cl[tr])
        acc.append(per_protein_r2(Pl[f], m.predict(Rl[f], cl[f]), Pl[tr].mean(0)))
    lag_r2[f"r2_lag{lag}"] = np.nanmean(acc, axis=0)

lag_tab = pd.DataFrame(lag_r2, index=list(pre_full.prot_features_))
lag_tab["lag_preference"] = lag_tab["r2_lag1"] - lag_tab["r2_lag0"]
lag_tab["lag_class"] = np.where(lag_tab["lag_preference"] >  0.05, "RNA-first / protein-later",
                       np.where(lag_tab["lag_preference"] < -0.05, "synchronous", "ambiguous"))
if USE_DEMO and truth is not None:
    lag_tab = lag_tab.join(truth["truth_class"])

lag_tab.sort_values("lag_preference", ascending=False).to_csv(OUT_DIR / "lag_atlas.csv")
print(lag_tab["lag_class"].value_counts().to_string())


In [ ]:
# ── Lag preference distribution ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# left: histogram
ax = axes[0]
pref = lag_tab["lag_preference"]
def safe_hist(ax, data, bins, **kw):
    if len(data) > 1 and data.std() > 1e-12:
        ax.hist(data, bins=min(bins, max(3, len(data)//2)), **kw)
safe_hist(ax, pref[pref < -0.05], 40, alpha=0.75, label="synchronous", color=sns.color_palette("Set2")[2])
safe_hist(ax, pref[pref.abs() <= 0.05], 20, alpha=0.75, label="ambiguous", color="lightgray")
safe_hist(ax, pref[pref > 0.05], 40, alpha=0.75, label="RNA-first / prot-later", color=sns.color_palette("Set2")[0])
ax.axvline(0, color="black", lw=1, ls="--")
ax.set_xlabel("Lag preference (R²_lag1 − R²_lag0)")
ax.set_ylabel("Proteins")
ax.set_title("Regulatory lag distribution", fontweight="bold")
ax.legend(fontsize=8); ax.spines[["top","right"]].set_visible(False)

# right: scatter R²_lag0 vs R²_lag1
ax = axes[1]
cls_col = {"RNA-first / protein-later": "steelblue", "synchronous": "seagreen", "ambiguous": "lightgray"}
for cls, g in lag_tab.groupby("lag_class"):
    ax.scatter(g["r2_lag0"].clip(-1,1), g["r2_lag1"].clip(-1,1),
               s=8, alpha=0.5, label=cls, color=cls_col.get(cls, "grey"))
lim = (-0.5, 1.0)
ax.plot(lim, lim, "k--", lw=1, label="y = x (no lag preference)")
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel("R² (RNA_t → protein_t)"); ax.set_ylabel("R² (RNA_t → protein_{t+1})")
ax.set_title("Lag-0 vs lag-1 predictability", fontweight="bold")
ax.legend(fontsize=7, markerscale=2); ax.spines[["top","right"]].set_visible(False)

plt.tight_layout()
plt.savefig(OUT_DIR / "figures" / "lag_atlas.png", dpi=150, bbox_inches="tight")
plt.show()

if USE_DEMO and truth is not None and "truth_class" in lag_tab:
    print()
    print("% RNA-first / protein-later by planted class:")
    print((lag_tab.assign(w=lag_tab["lag_class"]=="RNA-first / protein-later")
                  .groupby("truth_class")["w"].mean() * 100).round(1).to_string())


## 9 · Demo validation: recovery of planted classes

*(Only available when running on synthetic data.)*

Validates whether the pipeline actually detects what it claims to detect, before
you trust it on real data.


In [ ]:
if not USE_DEMO or truth is None:
    print("Skip — not running on demo data")
else:
    r2_truth = r2_table.join(truth["truth_class"])
    per_cls  = r2_truth.groupby("truth_class").median(numeric_only=True)
    per_cls["n"] = truth["truth_class"].value_counts()
    per_cls.to_csv(OUT_DIR / "r2_by_truth_class.csv")
    
    print("Pooled out-of-fold median R² by planted class:")
    display(per_cls.round(3))
    
    # ── radar / bar chart per class ──────────────────────────────────────
    fig, ax = plt.subplots(figsize=(11, 4.8))
    cls_list   = per_cls.index.tolist()
    model_list = [m for m in MODEL_ORDER if m in per_cls.columns]
    x = np.arange(len(cls_list))
    width = 0.13
    pal = dict(zip(model_list, sns.color_palette("Set2", len(model_list))))
    for i, m in enumerate(model_list):
        offset = (i - len(model_list)/2 + 0.5) * width
        ax.bar(x + offset, per_cls[m].clip(-0.5, 1), width,
               label=m, color=pal[m], alpha=0.85)
    ax.axhline(0, color="crimson", lw=1, ls="--")
    ax.set_xticks(x); ax.set_xticklabels(cls_list, rotation=18, ha="right")
    ax.set_ylabel("Median R²"); ax.set_title("Recovery by planted class", fontweight="bold")
    ax.legend(fontsize=7.5, ncol=2); ax.spines[["top","right"]].set_visible(False)
    plt.tight_layout()
    plt.savefig(OUT_DIR / "figures" / "recovery_by_class.png", dpi=150, bbox_inches="tight")
    plt.show()


## 10 · Output summary

In [ ]:
print(f"All outputs saved to: {OUT_DIR.resolve()}")
print()
files = sorted(OUT_DIR.rglob("*"))
csvs  = [f for f in files if f.suffix == ".csv"]
pngs  = [f for f in files if f.suffix == ".png"]
print(f"CSVs ({len(csvs)}):")
for f in csvs: print(f"  {f.relative_to(OUT_DIR)}")
print()
print(f"Figures ({len(pngs)}):")
for f in pngs: print(f"  {f.relative_to(OUT_DIR)}")


---
## What to do next

| Situation | Action |
|---|---|
| AE beats `pca_ridge` under `leave_variety_out` | Deep-learning story is defensible — consider ESM-2 embeddings (Stage 4) |
| AE ≈ `pca_ridge` | Report this — the RNA→protein map in your system is essentially linear |
| `design_only` ≈ AE | The model learned your design, not RNA biology — check preprocessing |
| Discordance hits clustered in worst-fit proteins | Model misspecification masquerading as regulation — use a better model for those proteins |
| Lag hits overlap heavily with discordance hits | Good: the same proteins respond post-transcriptionally AND with a delay |

**Cross-layer attribution (Question 3):** the gradient×input method in the pipeline
recovered cognate transcripts 0% of the time on synthetic data. For real cross-layer
wiring use sparse elastic net over pathway eigengenes, not AE gradients.
